# Support Vector Regression (SVR): California Housing Value Prediction

## Objective
Build and evaluate a Support Vector Regression model to predict `median_house_value` from California housing attributes.

This notebook includes:
- Data validation and exploratory data analysis (EDA)
- Leakage-safe preprocessing for numeric and categorical features
- A baseline SVR and cross-validation
- Target scaling using `TransformedTargetRegressor`
- Hyperparameter tuning with `GridSearchCV`
- Final test-set evaluation, baseline comparison, and residual analysis
- Saving the fitted preprocessing/model pipeline

**Important:** RBF SVR can be computationally expensive on larger datasets. Hyperparameter tuning is therefore performed on a reproducible sample of the training split by default. The held-out test split is kept separate from model selection.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

RANDOM_STATE = 42

# To keep RBF-SVR tuning practical on the California housing dataset.
# Increase this after confirming that runtime and memory are acceptable.
MAX_TUNING_ROWS = 3000
CV_SPLITS = 3

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("default")

## 2. Load the dataset

Expected project structure:

```text
project-root/
├── data/
│   └── housePriceDataset.csv
├── notebooks/
│   └── svr.ipynb
└── models/
```

The path lookup supports running this notebook from the project root or from the `notebooks/` directory.

In [ ]:
candidate_paths = [
    Path("data/housePriceDataset.csv"),
    Path("../data/housePriceDataset.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    expected = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        "Could not find housePriceDataset.csv. Check the dataset location. "
        f"Expected one of:\n{expected}"
    )

PROJECT_ROOT = DATA_PATH.resolve().parent.parent
df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

## 3. Inspect and validate the data

In [ ]:
display(df.info())
display(df.describe(include="all").T)

print("Duplicate rows:", df.duplicated().sum())

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_summary.sort_values("missing_percent", ascending=False))

In [ ]:
TARGET = "median_house_value"

FEATURES = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "ocean_proximity",
]

required_columns = FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(
        "The dataset is missing required columns: "
        f"{missing_columns}. Check the CSV schema."
    )

# Target values cannot be imputed for supervised training.
df = df.dropna(subset=[TARGET]).copy()

X = df[FEATURES].copy()
y = pd.to_numeric(df[TARGET], errors="raise").astype(float)

numerical_features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]
categorical_features = ["ocean_proximity"]

print("Feature columns:", FEATURES)
print(f"Target: {TARGET}")
print(f"Valid rows after removing missing targets: {len(df):,}")
print("Target summary:")
display(y.describe().to_frame(name=TARGET))

## 4. Exploratory data analysis

The plots below help inspect the target distribution, missing data, and relationships between selected features and house value. Correlation is descriptive and does not establish causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=df, x=TARGET, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of median house value")
axes[0].set_xlabel("Median house value")

missing_plot = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_percent")
if not missing_plot.empty:
    sns.barplot(
        x=missing_plot["missing_percent"],
        y=missing_plot.index,
        ax=axes[1],
    )
    axes[1].set_title("Missing values by feature")
    axes[1].set_xlabel("Missing values (%)")
    axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No missing values", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.scatterplot(
    data=df.sample(min(5000, len(df)), random_state=RANDOM_STATE),
    x="median_income",
    y=TARGET,
    alpha=0.35,
    s=18,
    ax=axes[0],
)
axes[0].set_title("Median income vs. median house value")

sns.boxplot(data=df, x="ocean_proximity", y=TARGET, ax=axes[1])
axes[1].set_title("House value by ocean proximity")
axes[1].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
numeric_for_corr = df.select_dtypes(include="number")

plt.figure(figsize=(10, 7))
sns.heatmap(
    numeric_for_corr.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1,
)
plt.title("Correlation matrix for numeric columns")
plt.tight_layout()
plt.show()

display(
    numeric_for_corr.corr()[TARGET]
    .sort_values(ascending=False)
    .rename("correlation_with_target")
    .to_frame()
)

## 5. Train-test split

The test set is held out for final evaluation. It is not used for preprocessing fit, cross-validation, or hyperparameter selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## 6. Preprocessing pipeline

- Numerical features: median imputation, then standardization.
- Categorical feature: most-frequent imputation, then one-hot encoding.
- The preprocessing is fitted within each training fold through the pipeline, preventing validation/test data from influencing imputation, scaling, or encoding.

SVR is sensitive to feature scales, so standardizing numerical inputs is important.

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
])

svr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("svr", SVR(kernel="rbf")),
])

svr_pipeline

## 7. Address target scale with TransformedTargetRegressor

`median_house_value` is measured in large dollar-like values, while SVR's default `epsilon=0.1` is tiny relative to that target scale. This can produce a poorly configured regression problem.

`TransformedTargetRegressor` standardizes the target during fitting and automatically converts predictions back to the original target scale. Because it is part of the estimator, the target scaler is fitted only on each training fold during cross-validation.

In [ ]:
model = TransformedTargetRegressor(
    regressor=svr_pipeline,
    transformer=StandardScaler(),
)

model

## 8. Create a reproducible tuning sample

RBF SVR can require substantial time and memory as the number of training examples grows. We sample from the training split for cross-validation and hyperparameter search. The sample is selected only from `X_train` / `y_train`; the test set remains untouched.

Increase `MAX_TUNING_ROWS` if you have enough time and memory. For large-scale production datasets, consider models that scale better than kernel SVR.

In [ ]:
tuning_size = min(MAX_TUNING_ROWS, len(X_train))

X_tune = X_train.sample(
    n=tuning_size,
    random_state=RANDOM_STATE,
)
y_tune = y_train.loc[X_tune.index]

print(f"Rows used for SVR tuning: {len(X_tune):,} of {len(X_train):,} available training rows")
print(f"Rows reserved for final test evaluation: {len(X_test):,}")

## 9. Baseline SVR cross-validation

This evaluates the initial RBF SVR with default SVR hyperparameters and target scaling. The split strategy is shuffled K-fold cross-validation, appropriate for this regression task.

In [ ]:
cv = KFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

baseline_cv_scores = cross_val_score(
    model,
    X_tune,
    y_tune,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
)

print("Baseline SVR CV R² scores:", np.round(baseline_cv_scores, 4))
print(f"Mean CV R²: {baseline_cv_scores.mean():.4f}")
print(f"Standard deviation: {baseline_cv_scores.std():.4f}")

## 10. Hyperparameter tuning with GridSearchCV

Tune `C`, `epsilon`, and `gamma`. Since the target is standardized inside `TransformedTargetRegressor`, candidate `epsilon` values are interpreted in standardized-target units.

The grid below is deliberately modest to keep the exercise feasible. The CV objective is \(R^2\), which is invariant to linear target scaling.

In [ ]:
param_grid = {
    "regressor__svr__C": [1, 10, 100],
    "regressor__svr__epsilon": [0.05, 0.2],
    "regressor__svr__gamma": ["scale", 0.1],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X_tune, y_tune)
best_model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)
print(f"\nBest mean CV R²: {grid_search.best_score_:.4f}")

In [ ]:
cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    .sort_values("rank_test_score")
    [[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "params",
    ]]
    .head(10)
    .reset_index(drop=True)
)
display(cv_results)

## 11. Final test-set evaluation

The selected model is evaluated once on the held-out test set. Metrics are reported on the original target scale after inverse transformation by `TransformedTargetRegressor`.

In [ ]:
y_pred = best_model.predict(X_test)

def regression_metrics(y_true, y_hat):
    mse = mean_squared_error(y_true, y_hat)
    return {
        "MAE": mean_absolute_error(y_true, y_hat),
        "Median absolute error": median_absolute_error(y_true, y_hat),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R²": r2_score(y_true, y_hat),
    }

test_metrics = pd.DataFrame.from_dict(
    regression_metrics(y_test, y_pred),
    orient="index",
    columns=["Score"],
)

display(test_metrics.style.format("{:,.4f}"))

## 12. Compare with a simple baseline

A useful regression model should outperform a simple baseline. The `DummyRegressor` below predicts the mean target value from the same tuning sample. The dummy model and the SVR are evaluated on exactly the same test rows.

In [ ]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_tune, y_tune)

dummy_pred = dummy_model.predict(X_test)

comparison_metrics = pd.DataFrame({
    "DummyRegressor (mean)": regression_metrics(y_test, dummy_pred),
    "Tuned SVR": regression_metrics(y_test, y_pred),
}).T

display(comparison_metrics.style.format("{:,.4f}"))

## 13. Visualize predictions and residuals

In [ ]:
plot_sample = pd.DataFrame({
    "Actual": np.asarray(y_test),
    "Predicted": np.asarray(y_pred),
    "Residual": np.asarray(y_test) - np.asarray(y_pred),
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=plot_sample.sample(min(4000, len(plot_sample)), random_state=RANDOM_STATE),
    x="Actual",
    y="Predicted",
    alpha=0.35,
    s=18,
    ax=axes[0],
)
low = min(plot_sample["Actual"].min(), plot_sample["Predicted"].min())
high = max(plot_sample["Actual"].max(), plot_sample["Predicted"].max())
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set_title("Actual vs. predicted")
axes[0].set_xlabel("Actual median house value")
axes[0].set_ylabel("Predicted median house value")

sns.scatterplot(
    data=plot_sample.sample(min(4000, len(plot_sample)), random_state=RANDOM_STATE),
    x="Predicted",
    y="Residual",
    alpha=0.35,
    s=18,
    ax=axes[1],
)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Residuals vs. predicted")
axes[1].set_xlabel("Predicted median house value")
axes[1].set_ylabel("Residual (actual - predicted)")

plt.tight_layout()
plt.show()

In [ ]:
display(
    plot_sample.head(20).style.format({
        "Actual": "{:,.0f}",
        "Predicted": "{:,.0f}",
        "Residual": "{:,.0f}",
    })
)

## 14. Save the fitted pipeline

The complete estimator stores preprocessing, SVR, and the fitted target transformer. Predictions are made with raw feature values in the same column format as `X`.

In [ ]:
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "svr_california_housing.joblib"
joblib.dump(best_model, model_path)

print(f"Saved model to: {model_path.resolve()}")

In [ ]:
# Example: reload and predict for a few passengers/records from the test features.
loaded_model = joblib.load(model_path)
sample_X = X_test.head(5)
sample_predictions = loaded_model.predict(sample_X)

display(
    sample_X.assign(Predicted_median_house_value=sample_predictions)
)

## 15. Conclusions and limitations

Complete this section after running all cells.

- **Best CV \(R^2\):** record `grid_search.best_score_`.
- **Test \(R^2\), MAE, and RMSE:** record the values from the test metrics table.
- **Baseline comparison:** assess whether tuned SVR improves on `DummyRegressor`, and by how much.
- **Residual analysis:** note whether errors vary by predicted value or show other visible patterns.
- **Data limitation:** California housing benchmark data has known limitations; median house values in this dataset are capped at an upper value. This can limit how well any model predicts expensive districts.
- **Runtime limitation:** tuning uses a random training subset because RBF SVR scales poorly as the number of rows grows. The measured model performance should be interpreted in light of that choice.
- **Further work:** compare with `RandomForestRegressor`, `HistGradientBoostingRegressor`, and a linear model using comparable validation splits. Do not choose a model based on the test set repeatedly.

A negative test \(R^2\) means the fitted model is worse than predicting the test-set mean under the \(R^2\) definition. Investigate the baseline comparison and residual plots rather than hiding poor results.